In [1]:
from pathlib import Path
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ============================================================
# EVALUATE PLEGMA UI/API CASE-STUDY PREDICTIONS FROM RAW CSVs
# ============================================================
#
# Expected structure:
#
# C:/Plegma_Programming/New_Evaluation/Histories168/
#   home01/
#       actual_target_day.csv
#       pred_no_history.csv
#       pred_with_history.csv
#
#   home04/
#       actual_target_day.csv
#       pred_no_history.csv
#       pred_with_history.csv
#
#   home07/
#       actual_target_day.csv
#       pred_no_history.csv
#       pred_with_history.csv
#
#   home05/
#       actual_target_day.csv
#       pred_no_history.csv
#       pred_with_history.csv
#
# Output created per home:
#
#   Evaluation_results/
#       evaluation_metrics.csv
#       hourly_comparison_all_models.csv
#       hourly_comparison_no_history.csv
#       hourly_comparison_with_history.csv
#       hourly_comparison_all_models.png
#       hourly_comparison_no_history.png
#       hourly_comparison_with_history.png
#       evaluation_metadata.json
#
# Global output:
#
#   C:/Plegma_Programming/New_Evaluation/Histories168/Evaluation_results/
#       all_homes_evaluation_metrics.csv
#       summary_by_scenario_and_model.csv
#       all_homes_hourly_comparison.csv
#
# Important:
#   - ensemble_weight_* columns are excluded
#   - no_history_ensemble and with_history_ensemble are kept as exported by UI/API
#   - no ensemble is recomputed here
#
# ============================================================


# ============================================================
# CONFIG
# ============================================================

ROOT_DIR = Path(r"C:/Plegma_Programming/New_Evaluation/SEASONAL")

SELECTED_HOMES = [
    "home01",
    "home04",
    "home07",
    "home05",
]

ACTUAL_FILE = "actual_target_day.csv"

# The script accepts alternative filenames too, in case the UI/API exports
# slightly different names.
PREDICTION_FILES = {
    "no_history": [
        "ui_pred_combined_no_history.csv",
        "pred_cold_start.csv",
        "pred_coldstart.csv",
    ],
    "with_history": [
        "ui_pred_combined_with_history.csv",
        "pred_withhistory.csv",
    ],
}

SAVE_ENCODING = "utf-8-sig"

TIME_COL = "timestamp"
ACTUAL_COL = "actual_consumption_Wh"

TIME_COL_CANDIDATES = [
    "timestamp",
    "datetime",
    "date_time",
    "time",
    "ds",
    "Timestamp",
    "DateTime",
]

ACTUAL_COL_CANDIDATES = [
    "consumption_Wh",
    "actual_consumption_Wh",
    "actual_Wh",
    "y_true",
    "actual",
    "target",
]

DAY_NAMES_GR = {
    0: "Δευτέρα",
    1: "Τρίτη",
    2: "Τετάρτη",
    3: "Πέμπτη",
    4: "Παρασκευή",
    5: "Σάββατο",
    6: "Κυριακή",
}


# ============================================================
# HELPERS
# ============================================================

def find_first_existing_col(df, candidates, required=True, label="column"):
    for c in candidates:
        if c in df.columns:
            return c

    lower_map = {str(c).lower(): c for c in df.columns}

    for c in candidates:
        if str(c).lower() in lower_map:
            return lower_map[str(c).lower()]

    if required:
        raise ValueError(
            f"Could not find {label}. Tried {candidates}. "
            f"Available columns: {list(df.columns)}"
        )

    return None


def maybe_convert_to_wh(series, col_name):
    values = pd.to_numeric(series, errors="coerce")
    name = str(col_name).lower()

    if "kwh" in name:
        values = values * 1000.0

    return values


def clean_name(name):
    name = str(name).strip()
    name = re.sub(r"[^A-Za-z0-9_]+", "_", name)
    name = re.sub(r"_+", "_", name).strip("_")

    name = name.replace("prediction_", "")
    name = name.replace("pred_", "")
    name = name.replace("_Wh", "")
    name = name.replace("_wh", "")

    return name


def standardize_timestamp(df, actual_timestamps=None, context="file"):
    df = df.copy()

    time_col = find_first_existing_col(
        df,
        TIME_COL_CANDIDATES,
        required=False,
        label=f"timestamp in {context}",
    )

    if time_col is not None:
        df[TIME_COL] = pd.to_datetime(df[time_col], errors="coerce")
        return df

    lower_cols = {str(c).lower(): c for c in df.columns}

    hour_col = lower_cols.get("hour")
    date_col = lower_cols.get("date") or lower_cols.get("target_date")

    if date_col is not None and hour_col is not None:
        date_values = pd.to_datetime(df[date_col], errors="coerce").dt.normalize()
        hour_values = pd.to_numeric(df[hour_col], errors="coerce")
        df[TIME_COL] = date_values + pd.to_timedelta(hour_values, unit="h")
        return df

    # Fallback: if prediction file has 24 rows and no timestamp,
    # use the timestamps from actual_target_day.csv.
    if actual_timestamps is not None and len(df) == len(actual_timestamps):
        df[TIME_COL] = list(pd.to_datetime(actual_timestamps))
        return df

    raise ValueError(
        f"Could not create timestamp for {context}. "
        f"Columns: {list(df.columns)}"
    )


def load_actual(actual_path):
    df = pd.read_csv(actual_path, low_memory=False)
    df = standardize_timestamp(df, context=str(actual_path))

    actual_col = find_first_existing_col(
        df,
        ACTUAL_COL_CANDIDATES,
        required=True,
        label=f"actual consumption column in {actual_path}",
    )

    actual = df[[TIME_COL, actual_col]].copy()
    actual = actual.rename(columns={actual_col: ACTUAL_COL})
    actual[ACTUAL_COL] = maybe_convert_to_wh(actual[ACTUAL_COL], actual_col)

    actual = (
        actual
        .dropna(subset=[TIME_COL, ACTUAL_COL])
        .groupby(TIME_COL, as_index=False)
        .agg({ACTUAL_COL: "mean"})
        .sort_values(TIME_COL)
        .reset_index(drop=True)
    )

    return actual


def is_prediction_column(col):
    c = str(col).lower()

    if c == TIME_COL.lower():
        return False

    if "weight" in c:
        return False

    exclude_terms = [
        "timestamp",
        "datetime",
        "date",
        "time",
        "hour",
        "home_id",
        "house_id",
        "target_date",
        "actual",
        "true",
        "consumption_wh",
        "energy_wh",
        "external_temperature",
        "internal_temperature",
        "temperature",
        "humidity",
        "residents",
        "rooms",
        "building_type",
        "occupation",
        "income",
        "heating",
        "water_heater",
        "appliance",
        "solar",
        "homeowner",
        "is_weekend",
        "is_holiday",
        "day_of_week",
        "month",
        "season",
        "remaining",
        "daily_total",
    ]

    if any(term in c for term in exclude_terms):
        return False

    include_terms = [
        "pred",
        "prediction",
        "forecast",
        "rf",
        "xgb",
        "lgbm",
        "ensemble",
    ]

    return any(term in c for term in include_terms)


def detect_prediction_columns(df):
    cols = []

    for col in df.columns:
        if not is_prediction_column(col):
            continue

        numeric = pd.to_numeric(df[col], errors="coerce")

        if numeric.notna().sum() > 0:
            cols.append(col)

    if cols:
        return cols

    # Fallback: if exactly one numeric non-time non-weight column exists
    numeric_candidates = []

    for col in df.columns:
        c = str(col).lower()

        if c == TIME_COL.lower() or "weight" in c:
            continue

        numeric = pd.to_numeric(df[col], errors="coerce")

        if numeric.notna().sum() > 0:
            numeric_candidates.append(col)

    if len(numeric_candidates) == 1:
        return numeric_candidates

    raise ValueError(
        "Could not detect prediction columns. "
        f"Numeric candidates: {numeric_candidates}. "
        f"All columns: {list(df.columns)}"
    )


def scenario_sort_key(col):
    order = {
        "rf": 0,
        "xgb": 1,
        "lgbm": 2,
        "ensemble": 3,
    }

    c = str(col).lower()

    for key, val in order.items():
        if key in c:
            return val

    return 99


def find_prediction_file(home_dir, filenames):
    for filename in filenames:
        path = home_dir / filename
        if path.exists():
            return path

    return None


def load_prediction_file(pred_path, scenario, actual_timestamps):
    df = pd.read_csv(pred_path, low_memory=False)

    df = standardize_timestamp(
        df,
        actual_timestamps=actual_timestamps,
        context=str(pred_path),
    )

    pred_cols = detect_prediction_columns(df)
    pred_cols = sorted(pred_cols, key=scenario_sort_key)

    out = df[[TIME_COL] + pred_cols].copy()

    rename_map = {}

    for col in pred_cols:
        base = clean_name(col)
        base_lower = base.lower()

        if base_lower in ["rf", "xgb", "lgbm", "ensemble"]:
            new_name = f"{scenario}_{base_lower}"
        elif "rf" in base_lower:
            new_name = f"{scenario}_rf"
        elif "xgb" in base_lower:
            new_name = f"{scenario}_xgb"
        elif "lgbm" in base_lower:
            new_name = f"{scenario}_lgbm"
        elif "ensemble" in base_lower:
            new_name = f"{scenario}_ensemble"
        else:
            new_name = f"{scenario}_{base}"

        rename_map[col] = new_name
        out[col] = maybe_convert_to_wh(out[col], col)

    out = out.rename(columns=rename_map)

    out = (
        out
        .groupby(TIME_COL, as_index=False)
        .mean(numeric_only=True)
        .sort_values(TIME_COL)
        .reset_index(drop=True)
    )

    prediction_labels = list(rename_map.values())

    preferred_order = [
        f"{scenario}_rf",
        f"{scenario}_xgb",
        f"{scenario}_lgbm",
        f"{scenario}_ensemble",
    ]

    prediction_labels = [
        c for c in preferred_order if c in prediction_labels
    ] + [
        c for c in prediction_labels if c not in preferred_order
    ]

    return out, prediction_labels


def mape_safe(y_true, y_pred, eps=1e-6):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    mask = y_true > eps

    if mask.sum() == 0:
        return np.nan

    return float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100)


def smape_safe(y_true, y_pred, eps=1e-6):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    denom = np.abs(y_true) + np.abs(y_pred) + eps

    return float(np.mean(2.0 * np.abs(y_pred - y_true) / denom) * 100)


def compute_metrics(df, pred_col):
    tmp = df[[TIME_COL, ACTUAL_COL, pred_col]].dropna().copy()

    if tmp.empty:
        return None

    y_true = tmp[ACTUAL_COL].astype(float).values
    y_pred = tmp[pred_col].astype(float).values

    actual_total_kWh = float(y_true.sum() / 1000.0)
    pred_total_kWh = float(y_pred.sum() / 1000.0)
    daily_abs_error_kWh = abs(pred_total_kWh - actual_total_kWh)

    actual_peak_idx = int(np.argmax(y_true))
    pred_peak_idx = int(np.argmax(y_pred))

    actual_peak_timestamp = pd.Timestamp(tmp.iloc[actual_peak_idx][TIME_COL])
    pred_peak_timestamp = pd.Timestamp(tmp.iloc[pred_peak_idx][TIME_COL])

    actual_peak_hour = int(actual_peak_timestamp.hour)
    pred_peak_hour = int(pred_peak_timestamp.hour)

    return {
        "rows": int(len(tmp)),

        "MAE_Wh": float(np.mean(np.abs(y_true - y_pred))),
        "RMSE_Wh": float(math.sqrt(np.mean((y_true - y_pred) ** 2))),
        "MAPE_%": mape_safe(y_true, y_pred),
        "sMAPE_%": smape_safe(y_true, y_pred),
        "bias_Wh": float(np.mean(y_pred - y_true)),

        "actual_total_kWh": actual_total_kWh,
        "pred_total_kWh": pred_total_kWh,
        "daily_abs_error_kWh": daily_abs_error_kWh,
        "daily_error_pct": (
            float(daily_abs_error_kWh / actual_total_kWh * 100.0)
            if actual_total_kWh > 0
            else np.nan
        ),

        "actual_peak_hour": actual_peak_hour,
        "pred_peak_hour": pred_peak_hour,
        "peak_hour_abs_error": abs(pred_peak_hour - actual_peak_hour),

        "actual_peak_Wh": float(np.max(y_true)),
        "pred_peak_Wh": float(np.max(y_pred)),
        "peak_value_abs_error_Wh": float(abs(np.max(y_pred) - np.max(y_true))),
    }


def scenario_from_col(col):
    c = str(col).lower()

    if c.startswith("no_history"):
        return "no_history"

    if c.startswith("with_history"):
        return "with_history"

    if c.startswith("cold_start") or c.startswith("coldstart"):
        return "no_history"

    if c.startswith("withhistory"):
        return "with_history"

    return "unknown"


def plot_comparison(df, pred_cols, out_path, title):
    plot_df = df.sort_values(TIME_COL).copy()
    x = plot_df[TIME_COL].dt.hour

    plt.figure(figsize=(13, 6))

    plt.plot(
        x,
        plot_df[ACTUAL_COL],
        marker="o",
        linewidth=2.6,
        label="Actual",
    )

    for col in pred_cols:
        if col not in plot_df.columns:
            continue

        plt.plot(
            x,
            plot_df[col],
            marker="o",
            linewidth=1.8,
            label=col,
        )

    plt.title(title)
    plt.xlabel("Hour of day")
    plt.ylabel("Consumption (Wh)")
    plt.xticks(range(24))
    plt.grid(True, alpha=0.3)
    plt.legend(fontsize=8, ncol=2)
    plt.tight_layout()
    plt.savefig(out_path, dpi=180, bbox_inches="tight")
    plt.close()


def get_home_dirs():
    return [ROOT_DIR / h for h in SELECTED_HOMES]


# ============================================================
# MAIN
# ============================================================

all_metrics = []
all_hourly = []

home_dirs = get_home_dirs()

print("=" * 140)
print("PLEGMA UI/API CASE-STUDY EVALUATION")
print("=" * 140)
print("Root:", ROOT_DIR)
print("Homes:", [p.name for p in home_dirs])
print()

for home_dir in home_dirs:
    home_id = home_dir.name

    results_dir = home_dir / "Evaluation_results"
    results_dir.mkdir(parents=True, exist_ok=True)

    actual_path = home_dir / ACTUAL_FILE

    print("=" * 140)
    print(f"Processing {home_id}")
    print("=" * 140)

    if not actual_path.exists():
        print(f"[SKIP] Missing actual file: {actual_path}")
        continue

    try:
        actual = load_actual(actual_path)
    except Exception as e:
        print(f"[SKIP] Could not load actual file for {home_id}: {e}")
        continue

    if actual.empty:
        print(f"[SKIP] Empty actual file for {home_id}")
        continue

    comparison = actual.copy()
    actual_timestamps = comparison[TIME_COL].tolist()

    prediction_cols_all = []
    prediction_cols_by_scenario = {}

    for scenario, filenames in PREDICTION_FILES.items():
        pred_path = find_prediction_file(home_dir, filenames)

        if pred_path is None:
            print(
                f"[WARN] Missing prediction file for {scenario}. "
                f"Tried: {filenames}"
            )
            continue

        try:
            pred_df, pred_labels = load_prediction_file(
                pred_path=pred_path,
                scenario=scenario,
                actual_timestamps=actual_timestamps,
            )
        except Exception as e:
            print(f"[WARN] Could not load {scenario}: {e}")
            continue

        comparison = comparison.merge(pred_df, on=TIME_COL, how="left")

        prediction_cols_all.extend(pred_labels)
        prediction_cols_by_scenario[scenario] = pred_labels

        print(
            f"{scenario}: loaded {len(pred_df)} rows "
            f"| file: {pred_path.name} "
            f"| columns: {pred_labels}"
        )

    if not prediction_cols_all:
        print(f"[SKIP] No prediction columns found for {home_id}")
        continue

    # Time/context columns
    comparison["home_id"] = home_id
    comparison["date"] = comparison[TIME_COL].dt.date
    comparison["hour"] = comparison[TIME_COL].dt.hour
    comparison["day_of_week"] = comparison[TIME_COL].dt.weekday
    comparison["day_name_gr"] = comparison["day_of_week"].map(DAY_NAMES_GR)
    comparison["is_weekend"] = (comparison["day_of_week"] >= 5).astype(int)

    # Save hourly comparison
    preferred_cols = [
        TIME_COL,
        "home_id",
        "date",
        "hour",
        "day_name_gr",
        "is_weekend",
        ACTUAL_COL,
    ] + prediction_cols_all

    preferred_cols = [c for c in preferred_cols if c in comparison.columns]

    hourly_path = results_dir / "hourly_comparison_all_models.csv"
    comparison[preferred_cols].to_csv(
        hourly_path,
        index=False,
        encoding=SAVE_ENCODING,
    )

    # Metrics
    home_metric_rows = []

    for pred_col in prediction_cols_all:
        metrics = compute_metrics(comparison, pred_col)

        if metrics is None:
            continue

        row = {
            "home_id": home_id,
            "target_date": str(comparison["date"].iloc[0]),
            "scenario": scenario_from_col(pred_col),
            "prediction_column": pred_col,
            **metrics,
        }

        home_metric_rows.append(row)
        all_metrics.append(row)

    metrics_df = pd.DataFrame(home_metric_rows)

    metrics_path = results_dir / "evaluation_metrics.csv"
    metrics_df.to_csv(
        metrics_path,
        index=False,
        encoding=SAVE_ENCODING,
    )

    # Scenario-specific hourly CSVs
    for scenario, cols in prediction_cols_by_scenario.items():
        scenario_cols = [
            TIME_COL,
            "home_id",
            "date",
            "hour",
            "day_name_gr",
            "is_weekend",
            ACTUAL_COL,
        ] + cols

        scenario_cols = [
            c for c in scenario_cols
            if c in comparison.columns
        ]

        scenario_path = results_dir / f"hourly_comparison_{scenario}.csv"

        comparison[scenario_cols].to_csv(
            scenario_path,
            index=False,
            encoding=SAVE_ENCODING,
        )

    # Plots
    plot_comparison(
        comparison,
        prediction_cols_all,
        results_dir / "hourly_comparison_all_models.png",
        f"{home_id} - Actual vs all predictions",
    )

    for scenario, cols in prediction_cols_by_scenario.items():
        plot_comparison(
            comparison,
            cols,
            results_dir / f"hourly_comparison_{scenario}.png",
            f"{home_id} - Actual vs {scenario} predictions",
        )

    # Metadata
    metadata = {
        "home_id": home_id,
        "actual_file": str(actual_path),
        "prediction_files": {
            scenario: [
                str(home_dir / filename)
                for filename in filenames
            ]
            for scenario, filenames in PREDICTION_FILES.items()
        },
        "results_dir": str(results_dir),
        "note": (
            "Evaluation created from raw actual/prediction CSVs. "
            "ensemble_weight_* columns were ignored because they are weights, not Wh predictions. "
            "UI/API-exported ensemble predictions were kept as-is."
        ),
        "prediction_columns_used": prediction_cols_all,
        "prediction_columns_by_scenario": prediction_cols_by_scenario,
        "ignored_columns_rule": "Columns containing 'weight' are excluded from metrics and plots.",
        "hourly_comparison_file": str(hourly_path),
        "metrics_file": str(metrics_path),
        "target_start": str(comparison[TIME_COL].min()),
        "target_end": str(comparison[TIME_COL].max()),
        "rows_actual": int(len(actual)),
        "rows_comparison": int(len(comparison)),
    }

    metadata_path = results_dir / "evaluation_metadata.json"

    with open(metadata_path, "w", encoding="utf-8") as f:
        json.dump(metadata, f, ensure_ascii=False, indent=2)

    print()
    print("Metrics:")

    if not metrics_df.empty:
        print(
            metrics_df[
                [
                    "scenario",
                    "prediction_column",
                    "rows",
                    "MAE_Wh",
                    "RMSE_Wh",
                    "MAPE_%",
                    "sMAPE_%",
                    "actual_total_kWh",
                    "pred_total_kWh",
                    "daily_abs_error_kWh",
                    "daily_error_pct",
                ]
            ].to_string(index=False)
        )

    print()
    print("Saved:")
    print("Hourly comparison:", hourly_path)
    print("Metrics:", metrics_path)
    print("Plots:", results_dir)
    print()

    all_hourly.append(comparison[preferred_cols].copy())


# ============================================================
# GLOBAL OUTPUTS
# ============================================================

global_results_dir = ROOT_DIR / "Evaluation_results"
global_results_dir.mkdir(parents=True, exist_ok=True)

if all_metrics:
    global_metrics = pd.DataFrame(all_metrics)

    global_metrics_path = global_results_dir / "all_homes_evaluation_metrics.csv"

    global_metrics.to_csv(
        global_metrics_path,
        index=False,
        encoding=SAVE_ENCODING,
    )

    summary = (
        global_metrics
        .groupby(["scenario", "prediction_column"], as_index=False)
        .agg(
            homes=("home_id", "nunique"),
            mean_MAE_Wh=("MAE_Wh", "mean"),
            mean_RMSE_Wh=("RMSE_Wh", "mean"),
            mean_MAPE_pct=("MAPE_%", "mean"),
            mean_sMAPE_pct=("sMAPE_%", "mean"),
            mean_bias_Wh=("bias_Wh", "mean"),
            mean_daily_abs_error_kWh=("daily_abs_error_kWh", "mean"),
            mean_daily_error_pct=("daily_error_pct", "mean"),
            mean_peak_value_abs_error_Wh=("peak_value_abs_error_Wh", "mean"),
        )
        .sort_values(["mean_RMSE_Wh", "mean_MAE_Wh"])
        .reset_index(drop=True)
    )

    summary_path = global_results_dir / "summary_by_scenario_and_model.csv"

    summary.to_csv(
        summary_path,
        index=False,
        encoding=SAVE_ENCODING,
    )

    print("=" * 140)
    print("GLOBAL SUMMARY BY SCENARIO / MODEL")
    print("=" * 140)
    print(summary.to_string(index=False))
    print()
    print("Saved global metrics:", global_metrics_path)
    print("Saved global summary:", summary_path)

else:
    print("[WARN] No metrics computed.")


if all_hourly:
    global_hourly = pd.concat(all_hourly, ignore_index=True)

    global_hourly_path = global_results_dir / "all_homes_hourly_comparison.csv"

    global_hourly.to_csv(
        global_hourly_path,
        index=False,
        encoding=SAVE_ENCODING,
    )

    print("Saved global hourly comparison:", global_hourly_path)


print("=" * 140)
print("DONE")
print("=" * 140)

PLEGMA UI/API CASE-STUDY EVALUATION
Root: C:\Plegma_Programming\New_Evaluation\SEASONAL
Homes: ['home01', 'home04', 'home07', 'home05']

Processing home01
no_history: loaded 24 rows | file: ui_pred_combined_no_history.csv | columns: ['no_history_rf', 'no_history_xgb', 'no_history_lgbm', 'no_history_ensemble']
with_history: loaded 24 rows | file: ui_pred_combined_with_history.csv | columns: ['with_history_rf', 'with_history_xgb', 'with_history_lgbm', 'with_history_ensemble']

Metrics:
    scenario     prediction_column  rows    MAE_Wh   RMSE_Wh    MAPE_%   sMAPE_%  actual_total_kWh  pred_total_kWh  daily_abs_error_kWh  daily_error_pct
  no_history         no_history_rf    24 22.756277 26.967973 21.833906 19.009638          2.493872        3.040023             0.546151        21.899703
  no_history        no_history_xgb    24 21.077223 28.181640 20.222013 17.280690          2.493872        2.999726             0.505853        20.283851
  no_history       no_history_lgbm    24 23.474830 4